## Step 1: Understand the data (Hetionet structure)

## Step 1: Read the data

In [1]:
# Parquet Format is easier and faster and save storage rather than CSV or TSV
import os
import pandas as pd
nodes_tsv = os.path.join("hetionet-data", "hetnet", "tsv", "hetionet-v1.0-nodes.tsv")
edges_gz = os.path.join("hetionet-data", "hetnet", "tsv", "hetionet-v1.0-edges.sif.gz")
nodes_parquet = os.path.join("hetionet-data", "hetnet", "tsv", "nodes.parquet")
edges_parquet = os.path.join("hetionet-data", "hetnet", "tsv", "edges.parquet")

try:
    print("Reading raw data files")
    nodes_df = pd.read_csv(nodes_tsv, sep="\t", low_memory=False)
    edges_df = pd.read_csv(edges_gz, sep="\t", compression="gzip", low_memory=False)
    
    print("Saving to Parquet format for ultimate speed")
    nodes_df.to_parquet(nodes_parquet, engine="fastparquet")
    edges_df.to_parquet(edges_parquet, engine="fastparquet")
    print("All files converted to Parquet successfully!")
    print("=" * 60)

    print("Unique Node kinds:")
    print(nodes_df['kind'].unique())
    print("\nUnique Edge metaedges:")
    print(edges_df['metaedge'].unique())
    print("-" * 60)
    
    print(f"🔹 First five nodes:\n{nodes_df.head()}\n")
    print(f"🔹 Pathway Nodes Example:\n{nodes_df[nodes_df['kind'] == 'Pathway'].head()}\n")
    print("-" * 60)
    print(f"🔹 First five edges:\n{edges_df.head()}\n")
    print(f"🔹 CrC Edges Example:\n{edges_df[edges_df['metaedge'] == 'CrC'].head()}\n")

except FileNotFoundError as e:
    print(f" Error: Please check the file path. System cannot find the file: {e}")
except Exception as e:
    print(f" Another error happened: {e}")

Reading raw data files
 Error: Please check the file path. System cannot find the file: [Errno 2] No such file or directory: 'hetionet-data\\hetnet\\tsv\\hetionet-v1.0-nodes.tsv'


## Dreamwalk Pipeline

In [2]:
import torch


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 الـ كود شغال حالياً على: {device} ({torch.cuda.get_device_name(0)})")

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
"""
DREAMwalk — Optimized, Leak-Free, GPU-Accelerated Pipeline
"""
import os
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy import sparse as sp #SparseGraph replace nx.Graph
from gensim.models import Word2Vec
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score
from sklearn.preprocessing import StandardScaler
import xgboost as xgb

import warnings
warnings.filterwarnings("ignore")

# Optional GPU backend for the similarity matrix multiply.
try:
    import cupy as cp
    import cupyx.scipy.sparse as csp
    GPU_AVAILABLE = True
except Exception:
    GPU_AVAILABLE = False

In [ ]:
# 1. CONFIGURATION (unchanged architecture)

CONFIG = {
 "walk_length": 100, # (100,)
 "num_walks": 20, #(10,20)
 "teleport_prob": 0.5,#(0.5,0.3)
 "p": 1.0, #DONT TOUCH UNLESS YOU KNOW WHAT YOU ARE DOING
 "q": 1.0, #DONT TOUCH UNLESS YOU KNOW WHAT YOU ARE DOING
 "embed_dim": 128,  # can change
 "window": 5,
 "min_count": 1,#
 "sg": 1,
 "workers": 4,
 "epochs": 5, ## early stop
 "n_folds": 10, 
 "xgb_params": {
 "n_estimators": 300,
 "max_depth": 6, #check overfitting 
 "learning_rate": 0.001,#(0.05,0.001)
 "subsample": 0.8,
 "colsample_bytree": 0.8,
 "eval_metric": "logloss",
 "random_state": 42,
 "n_jobs": -1,
 "tree_method": "hist",
 "device": "cuda", # used for GPU
 },
 "random_seed": 42,
 "cache_dir": "./dreamwalk_cache",
}

random.seed(CONFIG["random_seed"])
np.random.seed(CONFIG["random_seed"])
os.makedirs(CONFIG["cache_dir"], exist_ok=True)

In [1]:
# 2. DATA LOADING
# ─────────────────────────────────────────────
def load_hetionet_local():
    BASE_DIR = Path(__file__).resolve().parent.parent
    
    NODES_PATH = BASE_DIR / "hetionet-data" / "hetnet" / "tsv" / "nodes.parquet"
    EDGES_PATH = BASE_DIR / "hetionet-data" / "hetnet" / "tsv" / "edges.parquet"
    
    nodes_df = pd.read_parquet(NODES_PATH)
    edges_df = pd.read_parquet(EDGES_PATH)
    
    treats_df = edges_df[edges_df['metaedge'] == 'CtD'].copy()
    
    print(f"Nodes: {len(nodes_df):,}")
    print(f"Edges: {len(edges_df):,}")
    print(f"CtD Edges: {len(treats_df):,}")
    
    return nodes_df, edges_df, treats_df




In [ ]:
# 3. SPARSE GRAPH REPRESENTATION (replaces networkx.Graph)
# ─────────────────────────────────────────────
class SparseGraph:
    """
    Lightweight undirected graph backed by a SciPy CSR boolean adjacency
    matrix, plus padded neighbor arrays for vectorized random walking.
    Replaces nx.Graph for everything this pipeline needs.
    """
    def __init__(self, node_ids):
        self.node_ids = list(node_ids)
        self.id2idx = {n: i for i, n in enumerate(self.node_ids)}
        self.n = len(self.node_ids)
        self.adj = None          # CSR boolean, built via build_from_edges
        self.neighbor_idx = None  # (n, max_deg) int32, -1 padded
        self.neighbor_cnt = None  # (n,) int32

    def build_from_edges(self, src_idx, dst_idx):
        rows = np.concatenate([src_idx, dst_idx])
        cols = np.concatenate([dst_idx, src_idx])
        data = np.ones(len(rows), dtype=np.bool_)
        self.adj = sp.csr_matrix((data, (rows, cols)), shape=(self.n, self.n))
        self.adj.data[:] = True
        self.adj.sort_indices()
        self._build_padded_neighbors()

    def _build_padded_neighbors(self):
        indptr = self.adj.indptr
        indices = self.adj.indices
        deg = np.diff(indptr)
        max_deg = int(deg.max()) if len(deg) else 0
        max_deg = max(max_deg, 1)
        neighbor_idx = np.full((self.n, max_deg), -1, dtype=np.int32)
        for i in range(self.n):
            s, e = indptr[i], indptr[i + 1]
            neighbor_idx[i, : e - s] = indices[s:e]
        self.neighbor_idx = neighbor_idx
        self.neighbor_cnt = deg.astype(np.int32)

    def remove_edges(self, pairs_idx):
        """Return a NEW SparseGraph with the given (i, j) index pairs removed."""
        coo = self.adj.tocoo()
        remove_set = set()
        for i, j in pairs_idx:
            remove_set.add((i, j))
            remove_set.add((j, i))
        mask = np.array(
            [(r, c) not in remove_set for r, c in zip(coo.row, coo.col)]
        )
        new_adj = sp.csr_matrix(
            (coo.data[mask], (coo.row[mask], coo.col[mask])), shape=(self.n, self.n)
        )
        g = SparseGraph(self.node_ids)
        g.adj = new_adj
        g._build_padded_neighbors()
        return g


def build_graph(edges_df, nodes_df):
    node_ids = nodes_df['id'].tolist()
    G = SparseGraph(node_ids)
    src_idx = edges_df['source'].map(G.id2idx).to_numpy()
    dst_idx = edges_df['target'].map(G.id2idx).to_numpy()
    valid = (~pd.isna(src_idx)) & (~pd.isna(dst_idx))
    G.build_from_edges(src_idx[valid].astype(np.int64), dst_idx[valid].astype(np.int64))
    return G


def get_node_kinds(nodes_df):
    return dict(zip(nodes_df['id'], nodes_df['kind']))

In [ ]:
# 4. VECTORIZED JACCARD SIMILARITY (GPU via CuPy when available)
#    Computed strictly on G_train inside each fold -> no leakage.
# ─────────────────────────────────────────────
def jaccard_similarity_dense(adj_csr, idx_subset):
    """
    Dense Jaccard similarity for a (typically small, e.g. Compound or
    Disease) subset of nodes, computed via one matrix multiplication:
        intersection = A_sub @ A_sub.T
        union        = deg_i + deg_j - intersection
        jaccard      = intersection / union
    Runs on GPU (CuPy) if available, else NumPy/SciPy.
    """
    sub = adj_csr[idx_subset, :].astype(np.float32)  # (k, n)
    deg = np.asarray(sub.sum(axis=1)).ravel()         # (k,)

    if GPU_AVAILABLE:
        sub_gpu = csp.csr_matrix(sub)
        inter_gpu = sub_gpu.dot(sub_gpu.T)            # (k, k) sparse on GPU
        inter = cp.asnumpy(inter_gpu.toarray())
    else:
        inter = sub.dot(sub.T).toarray()

    union = deg[:, None] + deg[None, :] - inter
    with np.errstate(divide='ignore', invalid='ignore'):
        jac = np.where(union > 0, inter / union, 0.0)
    np.fill_diagonal(jac, 1.0)
    return jac.astype(np.float32)


def build_similarity_matrices(G, node_kinds, kind_a='Compound', kind_b='Disease'):
    """Returns dense jaccard matrices + the local node-id lists, all index-aligned."""
    drugs = [n for n in G.node_ids if node_kinds.get(n) == kind_a]
    diseases = [n for n in G.node_ids if node_kinds.get(n) == kind_b]
    drug_idx = np.array([G.id2idx[n] for n in drugs])
    disease_idx = np.array([G.id2idx[n] for n in diseases])

    drug_jac = jaccard_similarity_dense(G.adj, drug_idx) if len(drug_idx) else np.zeros((0, 0))
    disease_jac = jaccard_similarity_dense(G.adj, disease_idx) if len(disease_idx) else np.zeros((0, 0))

    # Pre-normalize rows to probability distributions (rows that sum to 0 stay 0).
    def row_normalize(mat):
        row_sum = mat.sum(axis=1, keepdims=True)
        out = np.zeros_like(mat)
        nz = row_sum[:, 0] > 0
        out[nz] = mat[nz] / row_sum[nz]
        return out

    drug_probs = row_normalize(drug_jac)
    disease_probs = row_normalize(disease_jac)

    return {
        "drugs": drugs, "diseases": diseases,
        "drug_idx": drug_idx, "disease_idx": disease_idx,
        "drug_probs": drug_probs, "disease_probs": disease_probs,
    }



**📄 Vectorized Jaccard Similarity (GPU-Accelerated)**

This section implements an efficient, leakage-safe, and hardware-adaptive method for computing pairwise Jaccard similarity matrices for biological node subsets (e.g., compounds and diseases). The design prioritizes vectorization, numerical stability, and optional GPU acceleration.

 **1. Dense Jaccard Similarity Function**
def jaccard_similarity_dense(adj_csr, idx_subset):

**Purpose**

Computes pairwise Jaccard similarity between nodes using a matrix-based formulation, avoiding slow pairwise Python loops.

**📐 Mathematical Formulation**

The similarity is computed as:

**Intersection:**

A⋅A
T

**Union:**

deg(i)+deg(j)−intersection

**Final Jaccard Score:**

J(i,j)=
union
intersection
	​

 **2. Subgraph Extraction**
sub = adj_csr[idx_subset, :].astype(np.float32)

 **What happens:**

Extracts only selected nodes (e.g., drugs or diseases)
Keeps full adjacency context for similarity computation
Converts matrix to float32 for performance

 **3. Degree Computation**

deg = np.asarray(sub.sum(axis=1)).ravel()

 **Purpose:**
Computes number of connections per node
Used to calculate the union term in Jaccard similarity

 **4. GPU Acceleration (Optional)**

**if GPU_AVAILABLE:**

 If GPU is available (CuPy):
Converts sparse matrix to CuPy CSR format
Performs matrix multiplication on GPU:
inter_gpu = sub_gpu.dot(sub_gpu.T)
Transfers result back to CPU (NumPy)

**If GPU is NOT available:**

Falls back to NumPy/SciPy CPU computation

**✔ Ensures hardware-independent execution**

 **5. Union Computation**

union = deg[:, None] + deg[None, :] - inter

 **What this does:**

Expands degree vector into a full matrix
Computes union for every node pair efficiently

 **6. Numerical Stability Handling**

with np.errstate(divide='ignore', invalid='ignore'):

 **Purpose:**

Prevents warnings from division by zero
Ensures stable execution
jac = np.where(union > 0, inter / union, 0.0)

 **Logic:**

If union > 0 → compute Jaccard
Else → assign 0

 **7. Self-Similarity Fix**

np.fill_diagonal(jac, 1.0)
Ensures every node has similarity = 1 with itself
Maintains mathematical correctness

 **8. Similarity Matrix Builder**

def build_similarity_matrices(G, node_kinds, kind_a='Compound', kind_b='Disease'):
 **Purpose**

Constructs structured similarity matrices for specific biological entity types.

 **Node Type Filtering**

drugs = [n for n in G.node_ids if node_kinds.get(n) == kind_a]
diseases = [n for n in G.node_ids if node_kinds.get(n) == kind_b]
Separates nodes by biological type:
Compounds (Drugs)
Diseases

 **Index Mapping**

drug_idx = np.array([G.id2idx[n] for n in drugs])
Converts node IDs → integer indices
Required for matrix-based computation

 **Jaccard Matrix Construction**

drug_jac = jaccard_similarity_dense(G.adj, drug_idx)
disease_jac = jaccard_similarity_dense(G.adj, disease_idx)
Computes similarity separately for:
Drugs
Diseases

 **Important:**

Computed inside each training fold only
Prevents data leakage

 **9. Row Normalization**

def row_normalize(mat):

 **Purpose**

Converts similarity scores into probability distributions

📌 Formula:
P(i,j)=
∑
j
	​

J(i,j)
J(i,j)


 **Steps:**

Computes row sums
Normalizes each row
Safely handles zero row

**10. Output Structure**

return {
    "drugs": drugs,
    "diseases": diseases,
    "drug_idx": drug_idx,
    "disease_idx": disease_idx,
    "drug_probs": drug_probs,
    "disease_probs": disease_probs,
}

 **Returned Components:**

Node lists (drugs, diseases)
Index mappings (for matrix operations)
Normalized similarity matrices (probability form)

**🧠 Final Insight**

This implementation achieves:

✔ Fully vectorized Jaccard computation (no loops)

✔ GPU acceleration via CuPy (optional)

✔ Numerical stability and safe execution

✔ Leakage-free design (fold-based computation)

✔ Probability-ready similarity outputs

In [ ]:
# 5. FULLY VECTORIZED RANDOM WALKS
#    All walks advance one step at a time, together, via NumPy ops.
# ─────────────────────────────────────────────
def vectorized_sample_from_probs(probs_matrix, rows, local_idx_lookup_list, rng):
    """
    For each entry in `rows` (a global-graph node index that is also a row
    in probs_matrix after lookup), sample a column index according to that
    row's probability distribution, via inverse-CDF (vectorized).
    Returns the sampled LOCAL column indices (into local_idx_lookup_list).
    Rows with all-zero probability return -1 (caller should fall back to graph walk).
    """
    cdf = np.cumsum(probs_matrix[rows], axis=1)          # (m, k)
    totals = cdf[:, -1]
    u = rng.random(len(rows)) * np.maximum(totals, 1e-12)
    sampled_cols = (cdf < u[:, None]).sum(axis=1)         # searchsorted, vectorized
    sampled_cols = np.clip(sampled_cols, 0, probs_matrix.shape[1] - 1)
    sampled_cols[totals <= 0] = -1
    return sampled_cols


def generate_all_walks_vectorized(G, num_walks, walk_length, teleport_prob,
                                   sim_data, node_kinds, p=1.0, q=1.0, seed=42):
    """
    Vectorized batched random walk generator with DREAMwalk teleportation.
    - All `num_walks * n_nodes` walks are advanced together, step by step.
    - Teleport candidates are drawn from the precomputed dense Compound/Disease
      Jaccard probability rows (sim_data), built on G_train only.
    - Graph steps use uniform sampling among neighbors (exact for p=q=1, the
      CONFIG default). For p!=1 or q!=1, a slower per-walk 2nd-order-biased
      fallback (`_biased_step_fallback`) is used automatically.
    """
    rng = np.random.default_rng(seed)
    n = G.n
    neighbor_idx = G.neighbor_idx
    neighbor_cnt = G.neighbor_cnt
    max_deg = neighbor_idx.shape[1]

    # Index lookups: global graph idx -> row in drug/disease prob matrices (-1 if N/A)
    drug_row_of = np.full(n, -1, dtype=np.int64)
    for local_i, gidx in enumerate(sim_data["drug_idx"]):
        drug_row_of[gidx] = local_i
    disease_row_of = np.full(n, -1, dtype=np.int64)
    for local_i, gidx in enumerate(sim_data["disease_idx"]):
        disease_row_of[gidx] = local_i

    drug_local_to_global = sim_data["drug_idx"]
    disease_local_to_global = sim_data["disease_idx"]

    base_order = np.arange(n)
    total_walks = num_walks * n
    walks = np.empty((total_walks, walk_length), dtype=np.int64)

    biased = (p != 1.0) or (q != 1.0)

    w = 0
    for _ in range(num_walks):
        rng.shuffle(base_order)
        walks[w:w + n, 0] = base_order
        w += n

    prev = np.full(total_walks, -1, dtype=np.int64)
    cur = walks[:, 0].copy()

    for step in range(1, walk_length):
        nxt = np.empty(total_walks, dtype=np.int64)

        is_drug = drug_row_of[cur] >= 0
        is_disease = disease_row_of[cur] >= 0
        teleport_roll = rng.random(total_walks) < teleport_prob
        teleport_drug = is_drug & teleport_roll
        teleport_disease = is_disease & teleport_roll & (~teleport_drug)
        graph_step = ~(teleport_drug | teleport_disease)

        if teleport_drug.any():
            idx = np.where(teleport_drug)[0]
            rows = drug_row_of[cur[idx]]
            local_cols = vectorized_sample_from_probs(sim_data["drug_probs"], rows, drug_local_to_global, rng)
            fallback = local_cols < 0
            sampled_global = np.where(fallback, -1, drug_local_to_global[np.clip(local_cols, 0, None)])
            nxt[idx[~fallback]] = sampled_global[~fallback]
            graph_step[idx[fallback]] = True  # fall back to graph step

        if teleport_disease.any():
            idx = np.where(teleport_disease)[0]
            rows = disease_row_of[cur[idx]]
            local_cols = vectorized_sample_from_probs(sim_data["disease_probs"], rows, disease_local_to_global, rng)
            fallback = local_cols < 0
            sampled_global = np.where(fallback, -1, disease_local_to_global[np.clip(local_cols, 0, None)])
            nxt[idx[~fallback]] = sampled_global[~fallback]
            graph_step[idx[fallback]] = True

        if graph_step.any():
            idx = np.where(graph_step)[0]
            cur_g = cur[idx]
            deg = neighbor_cnt[cur_g]
            dead = deg == 0
            if not biased:
                rnd = (rng.random(len(idx)) * np.maximum(deg, 1)).astype(np.int64)
                rnd = np.clip(rnd, 0, np.maximum(deg - 1, 0))
                chosen = neighbor_idx[cur_g, rnd]
            else:
                chosen = _biased_step_fallback(G, cur_g, prev[idx], rng, p, q)
            chosen[dead] = cur_g[dead]  # stuck node: stay in place
            nxt[idx] = chosen

        prev = cur
        cur = nxt
        walks[:, step] = cur

    return walks, G  # caller maps ids -> strings for Word2Vec


def _biased_step_fallback(G, cur_g, prev_g, rng, p, q):
    """Per-row (still vectorized over the batch, just not over neighbor lists)
    node2vec-style 2nd-order biased sampling, used only when p!=1 or q!=1."""
    out = np.empty(len(cur_g), dtype=np.int64)
    for i in range(len(cur_g)):
        c, pv = cur_g[i], prev_g[i]
        deg = G.neighbor_cnt[c]
        if deg == 0:
            out[i] = c
            continue
        neigh = G.neighbor_idx[c, :deg]
        if pv < 0:
            out[i] = neigh[rng.integers(0, deg)]
            continue
        weights = np.where(
            neigh == pv, 1.0 / p,
            np.where(np.isin(neigh, G.neighbor_idx[pv, :G.neighbor_cnt[pv]]), 1.0, 1.0 / q),
        )
        probs = weights / weights.sum()
        out[i] = rng.choice(neigh, p=probs)
    return out


def walks_to_str(walks, node_ids):
    id_arr = np.array(node_ids, dtype=object)
    return [list(map(str, id_arr[row])) for row in walks]


def train_embeddings(walks, config):
    model = Word2Vec(
        sentences=walks, vector_size=config['embed_dim'], window=config['window'],
        min_count=config['min_count'], sg=config['sg'], workers=config['workers'],
        epochs=config['epochs'], seed=config['random_seed'],
    )
    return model


def get_embedding(model, node_id):
    key = str(node_id)
    if key in model.wv:
        return model.wv[key]
    return np.zeros(model.vector_size)


def get_embeddings_batch(model, node_ids):
    dim = model.vector_size
    out = np.zeros((len(node_ids), dim), dtype=np.float32)
    for i, nid in enumerate(node_ids):
        key = str(nid)
        if key in model.wv:
            out[i] = model.wv[key]
    return out






**📄 Fully Vectorized Random Walks & Embeddings**

This section implements a high-performance random walk engine for heterogeneous graphs, combined with Word2Vec-based embedding learning. The design focuses on full vectorization, GPU-friendly probability sampling, and eliminating Python-level loops for scalability.

 **1. Vectorized Probability Sampling**

def vectorized_sample_from_probs(...)

 **Purpose**

Efficiently samples next nodes from probability distributions using a vectorized inverse CDF approach, avoiding loops entirely.


**Step 1: CDF Construction**

cdf = np.cumsum(probs_matrix[rows], axis=1)
Converts probabilities → cumulative distribution
Enables direct sampling without iteration

 **Step 2: Random Sampling**

u = rng.random(len(rows)) * np.maximum(totals, 1e-12)
Generates uniform random values per walk
Includes numerical stability safeguard (1e-12)

 **Step 3: Inverse CDF Selection**

sampled_cols = (cdf < u[:, None]).sum(axis=1)
Finds first index where CDF exceeds threshold
Fully vectorized replacement for searchsorted

**Output**

Returns selected column indices
Uses

 **1 for invalid or zero-probability rows**

 **2. Fully Vectorized Random Walk Generator**

def generate_all_walks_vectorized(...)

 **Goal**

Generate all random walks simultaneously, instead of per-node execution.

 **Core Idea**

Instead of:

node loop → step loop → sample neighbor

We do:

ALL walks updated together at each step

 **3. Initialization Phase**

 **RNG Setup**

rng = np.random.default_rng(seed)
Ensures reproducibility across runs

 **Graph Structures**

neighbor_idx, neighbor_cnt
Precomputed padded adjacency lists
Enable O(1) neighbor access

**Node Type Mapping**

drug_row_of, disease_row_of
Maps:
Global node → similarity matrix row
Used for teleportation decisions

 **4. Walk Initialization**

walks = np.empty((total_walks, walk_length))
Each node starts multiple independent walks
First step is randomly shuffled node assignment

**5. Main Random Walk Loop**

for step in range(1, walk_length):

At each step, all walks are updated in parallel.

 **Step 1: Node Type Detection**

is_drug, is_disease
Dynamically identifies node category
Enables type-aware transitions

**Step 2: Teleportation Decision**

teleport_roll = rng.random(...) < teleport_prob
Behavior:
Some walks:
Jump via similarity matrix (semantic move)
Others:
Continue graph traversal

 Adds global semantic exploration

**Step 3: Similarity-Based Teleportation**

vectorized_sample_from_probs(...)
Uses precomputed Jaccard similarity matrices
Applied separately for:
Drugs
Diseases

 **Ensures biologically meaningful jumps**

 **Step 4: Graph Transition**

neighbor_idx[cur_g, rnd]
Case A: Unbiased Walk (p = q = 1)
Uniform neighbor sampling
Fully vectorized
Case B: Biased Walk (Node2Vec-style)
_biased_step_fallback(...)

Uses:

p (return parameter) → controls backtracking
q (in-out parameter) → controls exploration depth

 **Step 5: Stuck Node Handling**

chosen[dead] = cur_g[dead]
If node has no neighbors:
Stay in place
Prevents invalid transitions

 **Step 6: Walk Update**

walks[:, step] = cur
Stores all walk positions simultaneously
Maintains full vectorized execution

 **6. Biased Random Walk Fallback**

def _biased_step_fallback(...)

 **Purpose**

Implements Node2Vec second-order transition rules

**Transition Rules**

For each candidate neighbor:

Return edge → weight = 1/p
Stay local → weight = 1
Explore outward → weight = 1/q

**Produces structured exploration behavior**

 **7. Conversion to Word2Vec Format**
def walks_to_str(...)
Converts numeric walks → string sequences
Required for Word2Vec training

**Example:**

[12, 45, 78] → ["12", "45", "78"]

 **8. Embedding Training (Word2Vec)**

def train_embeddings(...)

 **Purpose**

Learns node embeddings from walk sequences

 **Model Configuration**

Skip-gram (sg=1) → predicts context nodes
window → context size
vector_size → embedding dimension
epochs → training iterations

 **Output**

Dense vector representations of nodes
Capture:
Graph structure
Semantic relationships

 **9. Embedding Extraction**

Single Node
get_embedding(...)
Returns embedding vector
Defaults to zero vector if missing
Batch Extraction
get_embeddings_batch(...)
Efficient vectorized lookup
Used in downstream ML models (e.g., XGBoost)

**🧠 Final Insight**

This module achieves:

✔ Fully vectorized random walks (no loops)

✔ GPU-compatible probability sampling

✔ Hybrid graph + semantic teleportation

✔ Node2Vec-style biased exploration

✔ Scalable Word2Vec embedding learning

✔ High-performance batch embedding extraction

In [ ]:
# ─────────────────────────────────────────────
# 6. LEAK-FREE CROSS VALIDATION
#    Similarity matrices are rebuilt from G_train EVERY fold.
# ─────────────────────────────────────────────
def train_and_evaluate_clean(G, treats_df, node_kinds, config):
    drugs_all = [n for n in G.node_ids if node_kinds.get(n) == 'Compound']
    diseases_all = [n for n in G.node_ids if node_kinds.get(n) == 'Disease']

    pos_pairs = list(zip(treats_df['source'], treats_df['target']))
    pos_labels = [1] * len(pos_pairs)

    rng = np.random.default_rng(config['random_seed'])
    pos_set = set(pos_pairs)
    neg_pairs = []
    while len(neg_pairs) < len(pos_pairs):
        c = rng.choice(drugs_all)
        d = rng.choice(diseases_all)
        if (c, d) not in pos_set:
            neg_pairs.append((c, d))
    neg_labels = [0] * len(neg_pairs)

    all_pairs = np.array(pos_pairs + neg_pairs, dtype=object)
    all_labels = np.array(pos_labels + neg_labels)

    cv = StratifiedKFold(n_splits=config['n_folds'], shuffle=True, random_state=config['random_seed'])
    scaler = StandardScaler()

    aurocs, auprcs, accs = [], [], []
    results = []

    print(f"\n[eval] Starting {config['n_folds']}-fold CV WITHOUT Data Leakage "
          f"(similarity matrices rebuilt from G_train every fold)...")

    for fold, (train_idx, test_idx) in enumerate(cv.split(all_pairs, all_labels), 1):
        print(f"\n--- Processing Fold {fold}/{config['n_folds']} ---")
        pairs_train, pairs_test = all_pairs[train_idx], all_pairs[test_idx]
        y_train, y_test = all_labels[train_idx], all_labels[test_idx]

        # 1) Remove TEST-POSITIVE CtD edges from the graph -> G_train #update done
        import copy
        G_train = copy.deepcopy(G) if hasattr(G, 'copy') else G

        remove_idx_pairs = []
        for (u, v), y in zip(pairs_test, y_test):
            if y == 1 and u in G_train.id2idx and v in G_train.id2idx:
                remove_idx_pairs.append((G_train.id2idx[u], G_train.id2idx[v]))

        if remove_idx_pairs:
           res = G_train.remove_edges(remove_idx_pairs)
           if res is not None:
               G_train = res
        sim_data = build_similarity_matrices(G_train, node_kinds)
        # 3) Generate walks + embeddings on the clean graph only
        print(f"  [Fold {fold}] Generating vectorized random walks...")
        walks_idx, _ = generate_all_walks_vectorized(
            G_train, config['num_walks'], config['walk_length'], config['teleport_prob'],
            sim_data, node_kinds, config['p'], config['q'], seed=config['random_seed'] + fold,
        )
        walks_str = walks_to_str(walks_idx, G_train.node_ids)

        print(f"  [Fold {fold}] Training Word2Vec Embeddings...")
        embed_model_train = train_embeddings(walks_str, config)

        X_train = np.hstack([
            get_embeddings_batch(embed_model_train, pairs_train[:, 0]),  #error
            get_embeddings_batch(embed_model_train, pairs_train[:, 1]),
        ])
        X_test = np.hstack([
            get_embeddings_batch(embed_model_train, pairs_test[:, 0]),
            get_embeddings_batch(embed_model_train, pairs_test[:, 1]),
        ])

        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        clf = xgb.XGBClassifier(**config['xgb_params'])
        clf.fit(X_train_scaled, y_train, eval_set=[(X_test_scaled, y_test)], verbose=False)

        probs = clf.predict_proba(X_test_scaled)[:, 1]
        preds = clf.predict(X_test_scaled)

        auroc = roc_auc_score(y_test, probs)
        auprc = average_precision_score(y_test, probs)
        acc = accuracy_score(y_test, preds)

        aurocs.append(auroc); auprcs.append(auprc); accs.append(acc)
        results.append({'fold': fold, 'auroc': auroc, 'auprc': auprc, 'acc': acc})
        print(f"  --> Fold {fold} -> AUROC: {auroc:.4f} | AUPR: {auprc:.4f} | Acc: {acc:.4f}")

    print("\n" + "═" * 40)
    print(f"Mean Results (No Leakage): AUROC={np.mean(aurocs):.4f} | AUPR={np.mean(auprcs):.4f}")
    print("═" * 40)

    # Final production model: trained on the FULL graph (no test edges to hide
    # at this stage, since this model is for future, unlabeled, prediction).
    print("\nTraining final model on all data for future prediction...")
    sim_data_full = build_similarity_matrices(G, node_kinds)
    walks_idx_full, _ = generate_all_walks_vectorized(
        G, config['num_walks'], config['walk_length'], config['teleport_prob'],
        sim_data_full, node_kinds, config['p'], config['q'], seed=config['random_seed'],
    )
    walks_str_full = walks_to_str(walks_idx_full, G.node_ids)
    embed_model_all = train_embeddings(walks_str_full, config)

    X_all = np.hstack([
        get_embeddings_batch(embed_model_all, all_pairs[:, 0]),
        get_embeddings_batch(embed_model_all, all_pairs[:, 1]),
    ])
    X_all_scaled = scaler.fit_transform(X_all)

    final_clf = xgb.XGBClassifier(**config['xgb_params'])
    final_clf.fit(X_all_scaled, all_labels, verbose=False)

    return results, final_clf, scaler, embed_model_all, drugs_all, diseases_all, all_pairs, all_labels


**📄 Leak-Free Cross-Validation & Full Training Pipeline**

This section implements a strictly leak-free evaluation framework for the DREAMwalk model, followed by training a final production-ready classifier on the full dataset. The pipeline ensures reliable performance estimation and prevents any information leakage between training and testing.

 **1. Dataset Preparation**

 **Positive Samples (Ground Truth)**
pos_pairs = list(zip(treats_df['source'], treats_df['target']))
Extracts known Compound–Disease (CtD) interactions
Represents positive class (label = 1)
 **Negative Sampling**
Randomly generates (Compound, Disease) pairs
Ensures sampled pairs are not in the positive set

 **Result:**

Positive = real biological interactions
Negative = assumed non-interactions

 **Final Dataset Construction**

all_pairs = np.array(pos_pairs + neg_pairs)
all_labels = np.array(pos_labels + neg_labels)
Combines positive + negative samples
Produces a binary supervised learning dataset

 **2. Stratified Cross-Validation Setup**

cv = StratifiedKFold(n_splits=config['n_folds'])

**Purpose:**

Maintains class balance across folds
Ensures fair and stable evaluation

**3. Leak-Free Fold Processing**

Each fold follows a strict isolation pipeline.

 **Step 1 — Remove Test Edges (Critical Leakage Prevention)**
G_train = G.remove_edges(remove_idx_pairs)
Removes test CtD edges from graph
Prevents model from indirectly seeing test information

 **This is the core anti-leakage mechanism**

 **Step 2 — Fold-Specific Similarity Computation**

sim_data = build_similarity_matrices(G_train, node_kinds)
Jaccard similarity computed only on training graph
No reuse of global precomputed matrices

**✔ Ensures full training isolation**

 **Step 3 — Random Walk Generation**
generate_all_walks_vectorized(...)
Generates fully vectorized walks
Uses:
Graph structure (local transitions)
Similarity matrices (teleportation)

 **Output:**

Node sequence corpus for embedding training

 **Step 4 — Word2Vec Embedding Training**

train_embeddings(...)
Learns embeddings from walk corpus
Captures:
Structural relationships
Semantic similarity

 S**tep 5 — Feature Construction**

np.hstack([emb_u, emb_v])
Concatenates embeddings of (u, v)
Produces edge-level feature vectors

 **Step 6 — Feature Scaling**

scaler.fit_transform(X_train)
Normalizes feature space

 **Step 7 — XGBoost Training**

clf = xgb.XGBClassifier(...)
Trains classifier on:
Embedding-based features
Binary CtD labels

 **Step 8 — Evaluation Metrics**

ROC-AUC → ranking quality
AUPRC → performance on imbalanced data
Accuracy → classification correctness

 **Step 9 — Store Fold Results**

results.append(...)
Stores metrics per fold
Enables statistical performance analysis

 **4. Final Model Training (Production Phase)**

After cross-validation, a final model is trained on the entire dataset.

 **Step 1 — Full Graph Similarity**

sim_data_full = build_similarity_matrices(G, node_kinds)
Uses complete graph (no train/test split)

 **Step 2 — Full Random Walk Corpus**

generate_all_walks_vectorized(...)
Generates maximum coverage walk corpus

 **Step 3 — Final Embedding Model**

train_embeddings(...)
Produces final node embeddings for production use

**Step 4 — Full Feature Matrix**
X_all = np.hstack(...)
Builds complete Compound–Disease feature space

 **Step 5 — Final Classifier Training**

final_clf.fit(...)
Trained on all labeled data
Used for real-world prediction

 **5. Final Outputs**

return results, final_clf, scaler, embed_model_all, drugs_all, diseases_all

 **Returned Components:**

Cross-validation results (metrics per fold)
Final trained XGBoost model
Feature scaler
Final Word2Vec embedding model
Drug and disease node lists

**Key Contributions of the Pipeline**

 **1. Strict No-Leakage Design**

Test edges removed per fold
Similarity recomputed each time
Fully isolated training pipeline

 **2. End-to-End Learning Flow**

Graph → Random Walks → Embeddings → Features → Classifier

 **3. Production-Ready Model**

Final model trained on full dataset
Ready for inference on unseen pairs

 **4. Robust Evaluation**

Stratified K-Fold CV
Multiple metrics (AUROC, AUPRC, Accuracy)
Reliable performance estimation

**🧠 Final Insight**

This pipeline is a complete graph machine learning system that combines:

Structural graph learning
Semantic embedding extraction
Supervised classification
Strict leakage-free evaluation
Production deployment readiness

In [ ]:
# 7. VECTORIZED FULL PREDICTION (Compound x Disease matrix, no nested loop)
# ─────────────────────────────────────────────
def predict_all_pairs(drugs, diseases, final_clf, scaler, model, treats_df, top_k=20, batch_size=200_000):
    known = set(zip(treats_df['source'], treats_df['target']))
    drug_list = [d for d in drugs if str(d) in model.wv]
    disease_list = [d for d in diseases if str(d) in model.wv]
    print(f"\n[predict] Scoring {len(drug_list):,} x {len(disease_list):,} pairs (vectorized)...")

    drug_emb = get_embeddings_batch(model, drug_list)        # (D, dim)
    disease_emb = get_embeddings_batch(model, disease_list)  # (M, dim)

    D, M = len(drug_list), len(disease_list)
    drug_idx_grid, disease_idx_grid = np.meshgrid(np.arange(D), np.arange(M), indexing='ij')
    drug_idx_flat = drug_idx_grid.ravel()
    disease_idx_flat = disease_idx_grid.ravel()

    all_scores = np.empty(D * M, dtype=np.float32)
    total = D * M
    for start in tqdm(range(0, total, batch_size), desc="Scoring batches"):
        end = min(start + batch_size, total)
        d_idx = drug_idx_flat[start:end]
        s_idx = disease_idx_flat[start:end]
        X = np.hstack([drug_emb[d_idx], disease_emb[s_idx]])
        X_scaled = scaler.transform(X)
        all_scores[start:end] = final_clf.predict_proba(X_scaled)[:, 1]

    drug_arr = np.array(drug_list, dtype=object)[drug_idx_flat]
    disease_arr = np.array(disease_list, dtype=object)[disease_idx_flat]
    pred_df = pd.DataFrame({'compound': drug_arr, 'disease': disease_arr, 'score': all_scores})

    is_known = pred_df.apply(lambda r: (r['compound'], r['disease']) in known, axis=1)
    pred_df = pred_df[~is_known].sort_values('score', ascending=False)

    print(f"\n[predict] Top {top_k} novel drug repurposing candidates:")
    print(pred_df.head(top_k).to_string(index=False))
    return pred_df





**📄 Fully Vectorized Compound × Disease Prediction**
This section implements a fully vectorized prediction pipeline for scoring all possible Compound–Disease pairs in the DREAMwalk framework. The design avoids nested loops entirely, making large-scale biomedical inference faster, more scalable, and suitable for drug repurposing tasks.

**1) Input Filtering and Known Interaction Removal**

The pipeline first builds a set of all known Compound–Disease (CtD) interactions from the training data. These known links are later removed from the prediction results to ensure that the model only proposes novel candidate associations.

In addition, the drug and disease node lists are filtered so that only nodes with available embeddings in the trained Word2Vec model are retained. This guarantees that every predicted pair has a valid vector representation.

**2) Embedding Extraction**

Next, the model retrieves the learned embedding vectors for all valid compounds and diseases.

Drug embeddings are stored in a matrix of shape (D × embedding_dim)
Disease embeddings are stored in a matrix of shape (M × embedding_dim)

Each row corresponds to a biological entity represented in the learned latent space.

**3) Full Pairwise Candidate Generation**

To score every possible compound–disease combination, the pipeline constructs a complete Compound × Disease grid using vectorized indexing.

This replaces slow nested loops with a much more efficient approach, generating all candidate pairs in matrix form and flattening them into index arrays for downstream batch processing.

**4) Batch-Based Prediction**

Because the full pairwise space can be extremely large, prediction is performed in batches to remain memory efficient.

For each batch:

A subset of drug and disease indices is selected

Their embeddings are concatenated to form edge-level feature vectors

**[drug embedding∣disease embedding]**

The features are normalized using the same scaler fitted during training
The final XGBoost classifier predicts the probability of interaction for each pair

This design allows scalable inference without exceeding memory limits.

**5) Reconstruction of the Prediction Table**

After scoring all batches, the predicted probabilities are mapped back to their original biological entities. A final prediction table is then created with the following structure:

**Compound	Disease	Score**

This step restores the semantic meaning of the predictions by converting matrix indices back into node identifiers.

**6) Removal of Known Interactions**

All known drug–disease interactions already present in the original dataset are removed from the prediction table.

This is a critical step because it ensures that the model outputs only new, previously unseen candidate associations, rather than rediscovering known training examples.

**7) Ranking Candidate Associations**

The remaining predictions are sorted by their interaction probability scores in descending order.

As a result, the highest-confidence compound–disease pairs appear at the top of the table, making them easy to prioritize for downstream biological analysis or validation.

**8) Top-K Prediction Extraction**

Finally, the ranked table can be truncated to the Top-K predictions, returning the most promising candidate drug–disease associations.

These top-ranked predictions are particularly useful for:

Drug repurposing discovery
Biomedical hypothesis generation
Experimental validation planning

 **Final Output**

The function returns a DataFrame containing:

Predicted Compound–Disease pairs
Their corresponding interaction probability scores
Only novel candidates, after excluding known CtD interactions

 **Key Contributions of This Module
 Fully Vectorized Inference**

All possible Compound–Disease pairs are scored without nested loops, greatly reducing runtime.

 **Embedding-Based Prediction Space**

Predictions are made in a learned latent space, allowing the model to capture hidden biological relationships.

 **Memory-Safe Batch Processing**

Large pairwise prediction spaces are handled efficiently using batch-based inference.

 **Drug Repurposing Ready Output**

The final ranked list of novel drug–disease associations can be used directly in biomedical discovery workflows.

## Explainable AI (XAI) Module

This module provides a **complete SHAP-based explanation suite** for the trained XGBoost model, plus the existing biological metapath explainer.

| # | Plot | Purpose |
|---|------|---------|
| 1 | SHAP Summary Plot | Global — direction & magnitude of every feature |
| 2 | SHAP Bar Plot | Global — ranked mean \|SHAP\| per feature |
| 3 | SHAP Beeswarm Plot | Global — full distribution of per-sample impacts |
| 4 | SHAP Waterfall Plot | Local — why the top-1 candidate scored highest |
| 5 | SHAP Force Plot | Local — push/pull view for the top-1 candidate |
| 6 | SHAP Dependence Plot | Marginal effect of the most important feature |
| 7 | XGBoost Built-in Importance | Weight / Gain / Cover — side-by-side |

All figures are saved at **dpi=300** into `CONFIG['cache_dir']`.


In [ ]:
# ============================================================
# EXPLAINABLE AI (XAI) MODULE — Complete SHAP Suite
# ============================================================
# Uses REAL drug & disease names from nodes_df instead of
# generic Drug_Dim_X / Disease_Dim_X labels.
# Generates 7 publication-quality plots saved at dpi=300.
# ============================================================

import shap
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker


# ─────────────────────────────────────────────────────────────
# HELPER — Map embedding-dimension indices → readable labels
# e.g. "Drug_Dim_3" → "Aspirin · dim-3"
# ─────────────────────────────────────────────────────────────
def _make_feature_names(model, pred_df, nodes_df):
    """
    Build human-readable feature names for SHAP plots.

    Strategy
    --------
    The model embeds every node (drug or disease) as a vector of
    `dim` floats.  The XGBoost classifier receives
        [drug_dim_0 … drug_dim_{dim-1} | disease_dim_0 … disease_dim_{dim-1}]
    for every training pair.

    Instead of showing "Drug_Dim_42" we show the *actual name* of the
    most-frequently-occurring drug/disease in the training background,
    e.g. "Aspirin · dim-42" / "Lung cancer · dim-42".

    Returns
    -------
    feature_names : list[str]  length = 2*dim
    top_drug_name : str        most-common drug in top predictions
    top_disease_name : str     most-common disease in top predictions
    """
    id2name = dict(zip(nodes_df["id"], nodes_df["name"]))
    dim = model.vector_size

    # pick the most frequent drug & disease appearing in top predictions
    top_drug_id    = pred_df["compound"].value_counts().index[0]
    top_disease_id = pred_df["disease"].value_counts().index[0]
    top_drug_name    = id2name.get(top_drug_id,    top_drug_id)
    top_disease_name = id2name.get(top_disease_id, top_disease_id)

    feature_names = (
        [f"{top_drug_name} · dim-{i}"    for i in range(dim)] +
        [f"{top_disease_name} · dim-{i}" for i in range(dim)]
    )
    return feature_names, top_drug_name, top_disease_name


# ─────────────────────────────────────────────────────────────
# HELPER — Build a balanced background dataset for SHAP
# ─────────────────────────────────────────────────────────────
def _build_background(model, all_pairs, scaler, sample_size=200, seed=42):
    rng = np.random.default_rng(seed)
    n   = min(sample_size, len(all_pairs))
    idx = rng.choice(len(all_pairs), size=n, replace=False)
    pairs = all_pairs[idx]
    X = np.hstack([
        get_embeddings_batch(model, pairs[:, 0]),
        get_embeddings_batch(model, pairs[:, 1]),
    ])
    return scaler.transform(X)


# ─────────────────────────────────────────────────────────────
# HELPER — Extract SHAP values for the positive class
# ─────────────────────────────────────────────────────────────
def _shap_positive_class(explainer, X, feature_names):
    sv = explainer(X)

    if hasattr(sv, "values"):
        vals = sv.values
        if vals.ndim == 3:
            return shap.Explanation(
                values       = vals[:, :, 1],
                base_values  = sv.base_values[:, 1] if sv.base_values.ndim == 2
                               else sv.base_values,
                data         = sv.data,
                feature_names= feature_names,
            )
        sv.feature_names = feature_names
        return sv

    raw = explainer.shap_values(X)
    if isinstance(raw, list):
        return raw[1]
    return raw


# ─────────────────────────────────────────────────────────────
# MAIN XAI FUNCTION
# ─────────────────────────────────────────────────────────────
def explain_predictions(
    final_clf, scaler, model,
    pred_df,
    all_pairs, all_labels,
    nodes_df,                  # ← NEW: needed for real names
    top_k=10,
    background_size=200,
    seed=42,
):
    """
    Complete Explainable AI suite — all axis labels show REAL
    drug / disease names from Hetionet instead of generic indices.

    Plots saved to CONFIG['cache_dir'] at dpi=300:
      01_shap_summary_plot.png
      02_shap_bar_global_importance.png
      03_shap_beeswarm_plot.png
      04_shap_waterfall_top1.png
      05_shap_force_plot_top1.png
      06_shap_dependence_top_feature.png
      07_xgboost_builtin_importance.png
    """
    print("\n" + "=" * 62)
    print("  🧠  Explainable AI (XAI) — Complete SHAP Suite")
    print("=" * 62)

    out_dir = CONFIG["cache_dir"]
    os.makedirs(out_dir, exist_ok=True)

    # ── Real names ──────────────────────────────────────────────────
    id2name = dict(zip(nodes_df["id"], nodes_df["name"]))
    feature_names, top_drug_name, top_disease_name = _make_feature_names(
        model, pred_df, nodes_df
    )
    print(f"\n[XAI] Reference drug   : {top_drug_name}")
    print(f"[XAI] Reference disease : {top_disease_name}")

    # ── Build datasets ──────────────────────────────────────────────
    print("\n[XAI] Step 1/3 — Building background dataset …")
    X_bg = _build_background(model, all_pairs, scaler,
                              sample_size=background_size, seed=seed)

    print(f"[XAI] Step 2/3 — Building top-{top_k} prediction features …")
    top_pairs    = pred_df.head(top_k).copy()
    top_pairs["compound_name"] = top_pairs["compound"].map(id2name).fillna(top_pairs["compound"])
    top_pairs["disease_name"]  = top_pairs["disease"].map(id2name).fillna(top_pairs["disease"])

    X_top = np.hstack([
        get_embeddings_batch(model, top_pairs["compound"].tolist()),
        get_embeddings_batch(model, top_pairs["disease"].tolist()),
    ])
    X_top_scaled = scaler.transform(X_top)

    # ── Initialise SHAP ─────────────────────────────────────────────
    print("[XAI] Step 3/3 — Initialising SHAP TreeExplainer …")
    explainer = shap.TreeExplainer(
        final_clf,
        data                 = X_bg,
        feature_perturbation = "interventional",
    )

    print(f"[XAI] Computing SHAP values for background (n={background_size}) …")
    sv_bg  = _shap_positive_class(explainer, X_bg, feature_names)

    print(f"[XAI] Computing SHAP values for top {top_k} predictions …")
    sv_top = _shap_positive_class(explainer, X_top_scaled, feature_names)

    def _v(sv):
        return sv.values if hasattr(sv, "values") else sv

    def _save(fname, title=""):
        path = os.path.join(out_dir, fname)
        if title:
            plt.suptitle(title, fontsize=11, y=1.01)
        plt.tight_layout()
        plt.savefig(path, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"  ✅  {fname}")
        return path

    print("\n[XAI] Generating plots …\n")

    # ══════════════════════════════════════════════════════════
    # PLOT 1 — SHAP Summary Plot
    # Y-axis: real drug/disease names with dim index
    # ══════════════════════════════════════════════════════════
    plt.figure(figsize=(13, 9))
    shap.summary_plot(
        _v(sv_bg), X_bg,
        feature_names   = feature_names,
        max_display     = 20,
        show            = False,
        color_bar_label = "Feature Value (normalised)",
    )
    plt.xlabel("SHAP value (impact on model output)", fontsize=11)
    _save(
        "01_shap_summary_plot.png",
        f"SHAP Summary Plot  |  n={background_size} pairs  "
        f"|  drug ref: {top_drug_name}  |  disease ref: {top_disease_name}",
    )

    # ══════════════════════════════════════════════════════════
    # PLOT 2 — SHAP Bar Plot (Global Feature Importance)
    # ══════════════════════════════════════════════════════════
    plt.figure(figsize=(13, 9))
    shap.summary_plot(
        _v(sv_bg), X_bg,
        feature_names = feature_names,
        plot_type     = "bar",
        max_display   = 20,
        show          = False,
        color         = "steelblue",
    )
    plt.xlabel("Mean |SHAP value|", fontsize=11)
    _save(
        "02_shap_bar_global_importance.png",
        f"Global Feature Importance (mean |SHAP|)  "
        f"|  drug: {top_drug_name}  |  disease: {top_disease_name}",
    )

    # ══════════════════════════════════════════════════════════
    # PLOT 3 — SHAP Beeswarm Plot
    # ══════════════════════════════════════════════════════════
    try:
        plt.figure(figsize=(13, 9))
        if hasattr(sv_bg, "values"):
            shap.plots.beeswarm(sv_bg, max_display=20, show=False)
        else:
            exp_wrap = shap.Explanation(
                values        = _v(sv_bg),
                base_values   = np.zeros(len(X_bg)),
                data          = X_bg,
                feature_names = feature_names,
            )
            shap.plots.beeswarm(exp_wrap, max_display=20, show=False)
        _save(
            "03_shap_beeswarm_plot.png",
            f"SHAP Beeswarm  |  drug: {top_drug_name}  |  disease: {top_disease_name}",
        )
    except Exception as e:
        print(f"  ⚠️  Beeswarm skipped: {e}")
        plt.close()

    # ══════════════════════════════════════════════════════════
    # PLOT 4 — SHAP Waterfall Plot (top-1 real prediction)
    # Title shows the REAL compound ↔ disease names
    # ══════════════════════════════════════════════════════════
    best       = top_pairs.iloc[0]
    best_drug  = best["compound_name"]
    best_dis   = best["disease_name"]
    wf_title   = (
        f"SHAP Waterfall  |  {best_drug} ↔ {best_dis}"
        f"  (score = {best['score']:.4f})"
    )
    try:
        plt.figure(figsize=(14, 9))
        if hasattr(sv_top, "values"):
            shap.plots.waterfall(sv_top[0], max_display=20, show=False)
        else:
            base_val = (
                explainer.expected_value[1]
                if isinstance(explainer.expected_value, (list, np.ndarray))
                else explainer.expected_value
            )
            exp_top1 = shap.Explanation(
                values        = _v(sv_top)[0],
                base_values   = base_val,
                data          = X_top_scaled[0],
                feature_names = feature_names,
            )
            shap.plots.waterfall(exp_top1, max_display=20, show=False)
        _save("04_shap_waterfall_top1.png", wf_title)
    except Exception as e:
        print(f"  ⚠️  Waterfall skipped: {e}")
        plt.close()

    # ══════════════════════════════════════════════════════════
    # PLOT 5 — SHAP Force Plot (top-1)
    # ══════════════════════════════════════════════════════════
    try:
        base_val = (
            explainer.expected_value[1]
            if isinstance(explainer.expected_value, (list, np.ndarray))
            else explainer.expected_value
        )
        shap.initjs()
        shap.force_plot(
            base_val,
            _v(sv_top)[0],
            X_top_scaled[0],
            feature_names = feature_names,
            matplotlib    = True,
            show          = False,
        )
        fig = plt.gcf()
        fig.set_size_inches(18, 4)
        _save(
            "05_shap_force_plot_top1.png",
            f"SHAP Force Plot  |  {best_drug} ↔ {best_dis}",
        )
    except Exception as e:
        print(f"  ⚠️  Force plot skipped: {e}")
        plt.close()

    # ══════════════════════════════════════════════════════════
    # PLOT 6 — SHAP Dependence Plot (most important dimension)
    # Axis label shows the real drug/disease name + dim index
    # ══════════════════════════════════════════════════════════
    try:
        mean_abs     = np.abs(_v(sv_bg)).mean(axis=0)
        top_feat_idx = int(np.argmax(mean_abs))
        sec_feat_idx = int(np.argsort(mean_abs)[-2])

        plt.figure(figsize=(10, 7))
        shap.dependence_plot(
            top_feat_idx,
            _v(sv_bg),
            X_bg,
            feature_names     = feature_names,
            interaction_index = sec_feat_idx,
            show              = False,
            dot_size          = 25,
            alpha             = 0.65,
        )
        _save(
            "06_shap_dependence_top_feature.png",
            f"SHAP Dependence  |  {feature_names[top_feat_idx]}"
            f"  (colour = {feature_names[sec_feat_idx]})",
        )
    except Exception as e:
        print(f"  ⚠️  Dependence plot skipped: {e}")
        plt.close()

    # ══════════════════════════════════════════════════════════
    # PLOT 7 — XGBoost Built-in Feature Importance
    # Bar labels show real drug/disease names with dim index
    # ══════════════════════════════════════════════════════════
    try:
        importance_types = ["weight", "gain", "cover"]
        fig, axes = plt.subplots(1, 3, figsize=(22, 9))
        fig.suptitle(
            f"XGBoost Built-in Feature Importance (Top-20 per Type)\n"
            f"Drug ref: {top_drug_name}   |   Disease ref: {top_disease_name}",
            fontsize=13, fontweight="bold",
        )

        booster = final_clf.get_booster()
        viridis = plt.cm.viridis

        for ax, imp_type in zip(axes, importance_types):
            scores = booster.get_score(importance_type=imp_type)
            if not scores:
                ax.set_title(f"{imp_type.capitalize()}  (no data)")
                continue

            # fN → real feature name
            named = {
                (feature_names[int(k[1:])]
                 if k.startswith("f") and k[1:].isdigit() else k): v
                for k, v in scores.items()
            }
            items  = sorted(named.items(), key=lambda x: x[1], reverse=True)[:20]
            labels, values = zip(*items)

            colors = viridis(np.linspace(0.85, 0.25, len(labels)))
            ax.barh(
                range(len(labels)), list(reversed(values)),
                color=colors, edgecolor="white", linewidth=0.3,
            )
            ax.set_yticks(range(len(labels)))
            ax.set_yticklabels(list(reversed(labels)), fontsize=7)
            ax.set_xlabel(imp_type.capitalize(), fontsize=10)
            ax.set_title(f"Type: {imp_type.capitalize()}", fontsize=11, pad=8)
            ax.xaxis.set_major_formatter(
                ticker.FuncFormatter(
                    lambda x, _: f"{x:,.0f}" if x >= 1 else f"{x:.3f}"
                )
            )
            ax.spines[["top", "right"]].set_visible(False)

        _save("07_xgboost_builtin_importance.png")
    except Exception as e:
        print(f"  ⚠️  XGBoost importance plot skipped: {e}")
        plt.close()

    print("\n" + "=" * 62)
    print("  ✅  All XAI plots saved to: " + out_dir)
    print("=" * 62 + "\n")


In [ ]:
# ─────────────────────────────────────────────
# 8.(Metapath-Based Explanation)
# ─────────────────────────────────────────────

# دول أكواد العلاقات (metaedges) في الـ Hetionet اللي بتربط الدوا بالجينات، والمرض بالجينات
COMPOUND_GENE_EDGES = ['CbG', 'CuG', 'CdG']   # الدوا بيرتبط/بيرفع/بيقلل نشاط الجين
DISEASE_GENE_EDGES  = ['DaG', 'DuG', 'DdG']   # المرض مرتبط/بيرفع/بيقلل نشاط الجين
GENE_PATHWAY_EDGE    = 'GpPW'                  # الجين مشترك في مسار حيوي معين


def build_gene_lookup_tables(edges_df):
    """
    بنبني هنا 3 قواميس (dictionaries) هنستخدمهم بعد كده:
    1. كل دوا -> مجموعة الجينات اللي بيأثر فيها
    2. كل مرض -> مجموعة الجينات المرتبطة بيه
    3. كل جين -> مجموعة المسارات الحيوية اللي هو جزء منها
    """
    compound_to_genes = {}
    disease_to_genes = {}
    gene_to_pathways = {}

    cg_edges = edges_df[edges_df['metaedge'].isin(COMPOUND_GENE_EDGES)]
    for src, tgt in zip(cg_edges['source'], cg_edges['target']):
        compound_to_genes.setdefault(src, set()).add(tgt)

    dg_edges = edges_df[edges_df['metaedge'].isin(DISEASE_GENE_EDGES)]
    for src, tgt in zip(dg_edges['source'], dg_edges['target']):
        disease_to_genes.setdefault(src, set()).add(tgt)

    gp_edges = edges_df[edges_df['metaedge'] == GENE_PATHWAY_EDGE]
    for src, tgt in zip(gp_edges['source'], gp_edges['target']):
        gene_to_pathways.setdefault(src, set()).add(tgt)

    return compound_to_genes, disease_to_genes, gene_to_pathways


def get_shared_evidence(compound_id, disease_id, compound_to_genes, disease_to_genes, gene_to_pathways):
    """بترجع الجينات المشتركة والمسارات الحيوية المشتركة بين دوا ومرض معينين"""
    drug_genes = compound_to_genes.get(compound_id, set())
    disease_genes = disease_to_genes.get(disease_id, set())

    shared_genes = drug_genes & disease_genes

    drug_pathways = set()
    for g in drug_genes:
        drug_pathways |= gene_to_pathways.get(g, set())

    disease_pathways = set()
    for g in disease_genes:
        disease_pathways |= gene_to_pathways.get(g, set())

    shared_pathways = drug_pathways & disease_pathways
    return shared_genes, shared_pathways


def id_to_name_map(nodes_df):
    """تحويل أي معرف (id) لاسمه الحقيقي، مثلاً DB00945 يتحول لـ Aspirin"""
    return dict(zip(nodes_df['id'], nodes_df['name']))


def explain_via_metapaths(top_pairs, edges_df, nodes_df, top_n_genes=5, top_n_pathways=5):
    """
    لكل ترشيح من الترشيحات العالية، بنطلع الجينات المشتركة والمسارات الحيوية المشتركة
    بين الدوا والمرض، عشان نديله تفسير بيولوجي مفهوم مش بس أرقام إحصائية
    """
    print("\n" + "="*50)
    print(" 🧬 التفسير البيولوجي عن طريق المسارات المشتركة")
    print("="*50)

    compound_to_genes, disease_to_genes, gene_to_pathways = build_gene_lookup_tables(edges_df)
    name_of = id_to_name_map(nodes_df)

    records = []
    for _, row in top_pairs.iterrows():
        c_id, d_id, score = row['compound'], row['disease'], row['score']
        shared_genes, shared_pathways = get_shared_evidence(
            c_id, d_id, compound_to_genes, disease_to_genes, gene_to_pathways
        )

        gene_names = [name_of.get(g, g) for g in list(shared_genes)[:top_n_genes]]
        pathway_names = [name_of.get(p, p) for p in list(shared_pathways)[:top_n_pathways]]

        records.append({
            'compound': name_of.get(c_id, c_id),
            'disease': name_of.get(d_id, d_id),
            'score': score,
            'n_shared_genes': len(shared_genes),
            'shared_genes_sample': gene_names,
            'n_shared_pathways': len(shared_pathways),
            'shared_pathways_sample': pathway_names,
        })

        print(f"\n💊 {name_of.get(c_id, c_id)}  ←→  🦠 {name_of.get(d_id, d_id)}  (Score: {score:.4f})")
        print(f"   عدد الجينات المشتركة: {len(shared_genes)}  | أمثلة: {gene_names}")
        print(f"   عدد المسارات الحيوية المشتركة: {len(shared_pathways)}  | أمثلة: {pathway_names}")

    evidence_df = pd.DataFrame(records)
    evidence_path = os.path.join(CONFIG['cache_dir'], "metapath_evidence.csv")
    evidence_df.to_csv(evidence_path, index=False)
    print(f"\n✅ جدول الأدلة البيولوجية اتسيف هنا: {evidence_path}")
    print("="*50 + "\n")
    return evidence_df


# ─────────────────────────────────────────────
# 9. التفسير العام الحقيقي (True Global Explanation)
# ─────────────────────────────────────────────

def explain_global(final_clf, scaler, model, all_pairs, all_labels, sample_size=200, seed=42):
    """
    التفسير العام الحقيقي: بياخد عينة عشوائية فيها حالات ناجحة (positive)
    وحالات فاشلة (negative) مع بعض، مش بس أعلى النتائج
    """
    print("\n" + "="*50)
    print(" 🌍 التفسير العام الحقيقي (عينة عشوائية، ناجح وفاشل مع بعض)")
    print("="*50)

    rng = np.random.default_rng(seed)
    n_sample = min(sample_size, len(all_pairs))
    idx = rng.choice(len(all_pairs), size=n_sample, replace=False)
    sample_pairs = all_pairs[idx]

    X = np.hstack([
        get_embeddings_batch(model, sample_pairs[:, 0]),
        get_embeddings_batch(model, sample_pairs[:, 1]),
    ])
    X_scaled = scaler.transform(X)

    dim = model.vector_size
    feature_names = [f"Drug_Dim_{i}" for i in range(dim)] + [f"Disease_Dim_{i}" for i in range(dim)]

    explainer = shap.TreeExplainer(final_clf)
    shap_values = explainer(X_scaled)
    shap_values.feature_names = feature_names

    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_scaled, feature_names=feature_names, show=False)
    plt.title(f"التفسير العام الحقيقي لأهمية الخصائص (عينة عشوائية n={n_sample})", fontsize=14)
    plt.tight_layout()
    global_plot_path = os.path.join(CONFIG['cache_dir'], "shap_true_global.png")
    plt.savefig(global_plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ رسمة التفسير العام اتسيفت هنا: {global_plot_path}")
    print("="*50 + "\n")
    return shap_values

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def save_pipeline_plots(results, pred_df, config, nodes_df=None):
    """
    Generates and saves research-grade plots with REAL drug/disease names.
    1. CV Performance Bar Plot
    2. Prediction Scores Distribution (Density Plot) — top candidates labelled
    """
    print("\n📊 Generating and saving pipeline evaluation plots...")

    # Build real-name lookup if nodes_df is provided
    id2name = dict(zip(nodes_df["id"], nodes_df["name"])) if nodes_df is not None else {}

    res_df = pd.DataFrame(results)

    # ─── Plot 1: CV Metrics across Folds ───────────────────────
    plt.figure(figsize=(9, 6))
    melted_df = res_df.melt(
        id_vars='fold',
        value_vars=['auroc', 'auprc', 'acc'],
        var_name='Metric', value_name='Value'
    )
    sns.barplot(data=melted_df, x='Metric', y='Value',
                errorbar="sd", palette="viridis")
    plt.title("Model Performance Metrics across 10-Folds\n"
              "DREAMwalk — Drug Repurposing Pipeline", fontsize=12)
    plt.ylim(0, 1.05)
    plt.ylabel("Score")
    plt.xlabel("Evaluation Metric")
    plot1_path = os.path.join(config['cache_dir'], "cv_metrics_performance.png")
    plt.savefig(plot1_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Performance Plot saved to: {plot1_path}")

    # ─── Plot 2: Score Distribution with real top-candidate labels ─
    plt.figure(figsize=(10, 6))
    sns.kdeplot(pred_df['score'], fill=True, color="steelblue", bw_adjust=0.5,
                label="All novel pairs")
    plt.axvline(x=0.5, color='red', linestyle='--', label='Threshold (0.5)')

    # Annotate top-5 candidates with their REAL names
    top5 = pred_df.head(5)
    for _, row in top5.iterrows():
        drug_name    = id2name.get(row['compound'], row['compound'])
        disease_name = id2name.get(row['disease'],  row['disease'])
        label = f"{drug_name}\n↔ {disease_name}"
        plt.axvline(x=row['score'], color='green', linestyle=':', alpha=0.6)
        plt.text(row['score'] + 0.002, plt.ylim()[1] * 0.85,
                 label, fontsize=6, color='darkgreen',
                 ha='left', va='top', rotation=90)

    plt.title("Distribution of Repurposing Scores — Novel Drug–Disease Pairs\n"
              "(green lines = top-5 candidates)", fontsize=12)
    plt.xlabel("Predicted Repurposing Probability (Score)")
    plt.ylabel("Density")
    plt.legend()
    plot2_path = os.path.join(config['cache_dir'], "predictions_distribution.png")
    plt.savefig(plot2_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Prediction Distribution Plot saved to: {plot2_path}")

    # ─── Plot 3: Top-20 Candidates Bar Chart with REAL names ────
    plt.figure(figsize=(12, 8))
    plot_df = pred_df.head(20).copy()
    plot_df["label"] = (
        plot_df["compound"].map(id2name).fillna(plot_df["compound"]) + "\n↔ " +
        plot_df["disease"].map(id2name).fillna(plot_df["disease"])
    )
    colors = plt.cm.viridis(
        [s / plot_df["score"].max() for s in plot_df["score"]]
    )
    plt.barh(plot_df["label"][::-1], plot_df["score"][::-1],
             color=colors[::-1], edgecolor="white", linewidth=0.4)
    plt.xlabel("Repurposing Score", fontsize=11)
    plt.title("Top-20 Drug Repurposing Candidates\n"
              "DREAMwalk — Hetionet Knowledge Graph", fontsize=12)
    plt.xlim(0, 1.05)
    for i, (_, row) in enumerate(plot_df[::-1].iterrows()):
        plt.text(row["score"] + 0.01, i, f'{row["score"]:.3f}',
                 va='center', fontsize=8)
    plt.tight_layout()
    plot3_path = os.path.join(config['cache_dir'], "top20_candidates.png")
    plt.savefig(plot3_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Top-20 Candidates Plot saved to: {plot3_path}")


In [2]:
# ============================================================
# MAIN PIPELINE
# ============================================================

def main():
    print("=" * 55)
    print("  DREAMwalk Optimized Pipeline (Leak-Free + Vectorized)")
    print(f"  GPU similarity backend (CuPy) available: {GPU_AVAILABLE}")
    print("=" * 55)

    nodes_df, edges_df, treats_df = load_hetionet_local()
    node_kinds = get_node_kinds(nodes_df)
    G = build_graph(edges_df, nodes_df)

    (results, final_clf, scaler, embed_model_all,
     drugs, diseases, all_pairs, all_labels) = train_and_evaluate_clean(
        G, treats_df, node_kinds, CONFIG
    )

    results_df = pd.DataFrame(results)
    results_df.to_csv(os.path.join(CONFIG['cache_dir'], "clean_cv_results.csv"), index=False)

    pred_df = predict_all_pairs(
        drugs, diseases, final_clf, scaler, embed_model_all, treats_df, top_k=20
    )
    pred_df.to_csv(os.path.join(CONFIG['cache_dir'], "clean_predictions.csv"), index=False)

    # ── 1  Complete XAI suite (7 SHAP + XGBoost plots) ──────────
    # nodes_df is now passed so all labels show REAL names
    explain_predictions(
        final_clf, scaler, embed_model_all,
        pred_df, all_pairs, all_labels,
        nodes_df=nodes_df,          # ← real drug/disease names
        top_k=10,
        background_size=200,
    )

    # ── 2  Biological metapath explanation ──────────────────────
    explain_via_metapaths(pred_df.head(10), edges_df, nodes_df)

    # ── 3  CV performance + score distribution plots ────────────
    save_pipeline_plots(results, pred_df, CONFIG, nodes_df)

    print("\n[done] Optimized, leak-free DREAMwalk complete.")


if __name__ == "__main__":
    main()


  DREAMwalk Optimized Pipeline (Leak-Free + Vectorized)


NameError: name 'GPU_AVAILABLE' is not defined


**Fully Vectorized Compound × Disease Prediction**
This section implements a highly optimized inference pipeline that evaluates all possible Compound–Disease pairs in a fully vectorized way. The design avoids nested loops entirely and is built for large-scale biomedical prediction and drug repurposing.

 **1. Input Filtering & Known Interaction Removal**

known = set(zip(treats_df['source'], treats_df['target']))

 **Purpose**

Stores all known Compound–Disease (CtD) interactions
Used to exclude existing relationships from predictions

 **Valid Node Filtering**

drug_list = [d for d in drugs if str(d) in model.wv]

 **Logic:**
Keeps only nodes present in the Word2Vec vocabulary
Ensures all prediction inputs have valid embeddings

**✔Output:**

Clean drug list
Clean disease list

**2.Embedding Extraction**

drug_emb = get_embeddings_batch(model, drug_list)

 **Purpose**
Converts node IDs → embedding vectors

**📐Shapes:**

Drugs → (D × embedding_dim)
Diseases → (M × embedding_dim)


**Each row represents a biological entity in latent semantic space**

 **3. Full Pairwise Grid Construction**

 **Purpose**
Generates all possible Compound × Disease combinations

 **Key Advantage:**

Replaces nested loops with vectorized indexing
Produces D × M candidate pairs efficiently

 **4.Batch-Based Prediction Strategy**

for start in range(0, total, batch_size):

 **Why batching?**

Prevents memory overflow
Enables scalable inference over large graphs

 **4.1Pair Selection**

d_idx = drug_idx_flat[start:end]
s_idx = disease_idx_flat[start:end]
Selects subset of pairs per batch

**4.2 Feature Construction**

X = np.hstack([drug_emb[d_idx], disease_emb[s_idx]])

 **Meaning:**

Each pair becomes:

[drug_embedding | disease_embedding]

**📐Shape:**

(batch_size × 2 × embedding_dim)

 **4.3 Feature Scaling**

X_scaled = scaler.transform(X)

 **Purpose:**
Ensures consistency with training distribution
Critical for stable XGBoost predictions

**4.4 Model Inference**

final_clf.predict_proba(X_scaled)
Outputs probability of interaction for each pair
Stored in all_scores

 **5.Reconstruction of Prediction Table**

pred_df = pd.DataFrame(...)

**Purpose**

Converts numerical predictions into interpretable format:

| compound | disease | score |

 **Index Mapping**

drug_arr = np.array(drug_list)[drug_idx_flat]
Converts indices → original biological IDs
Restores semantic meaning of predictions

 **6. Removal of Known Interactions**

pred_df = pred_df[~is_known]

 **Purpose:**

Removes existing CtD edges
Ensures only novel predictions remain

 **Prevents trivial re-discovery of training data**

 **7.Ranking Predictions**

pred_df.sort_values('score', ascending=False)

 **Behavior:**

Highest probability pairs appear first
Enables prioritization for biological validation

 **8.Top-K Candidate Extraction**

pred_df.head(top_k)

 **Purpose:**

Returns top predicted interactions for:

 Drug repurposing
 Biological hypothesis generation
 Experimental validation

 **9.Final Output**

return pred_df
 **Output Contains:**

All predicted Compound–Disease pairs
Interaction probability scores
Filtered novel candidates

 **Key Innovations**

 **1. Fully Vectorized Prediction**

No loops
O(D × M) operations handled efficiently

 **2.Embedding-Based Latent Space Prediction**

Predictions performed in learned vector space
Captures hidden biological relationships

 **3.Batch Inference Engine**

Memory-safe design
Scales to large biomedical graphs

**4.Drug Repurposing Ready Output**

Ranked novel candidates
Directly usable for discovery pipelines

**🧠 Final Insight**
This module transforms the DREAMwalk model into a real-world biomedical prediction engine by:

Generating all possible drug–disease pairs efficiently
Scoring them in embedding space
Filtering known interactions
Producing ranked novel hypotheses

In [ ]:
import pandas as pd
print(pd.read_csv("dreamwalk_cache/clean_cv_results.csv"))

   fold     auroc     auprc       acc
0     1  0.926316  0.936316  0.887417
1     2  0.913509  0.913556  0.834437
2     3  0.939123  0.935705  0.887417
3     4  0.958772  0.962869  0.894040
4     5  0.954035  0.955006  0.867550
5     6  0.948070  0.951950  0.894040
6     7  0.927895  0.916011  0.854305
7     8  0.947719  0.951629  0.880795
8     9  0.957368  0.950321  0.907285
9    10  0.923684  0.904588  0.867550


In [ ]:
import pandas as pd

df = pd.read_csv("dreamwalk_cache/clean_cv_results.csv")

print("--- Full Cross-Validation Results ---")
print(df)
print("-" * 50)

print("--- Overall Average Scores ---")
 
metrics_cols = [col for col in ['AUROC', 'AUPR', 'Acc', 'Accuracy'] if col in df.columns]

if metrics_cols:
    averages = df[metrics_cols].mean()
    for metric, val in averages.items():
        print(f"Average {metric}: {val:.4f}")
else:
    print(df.mean(numeric_only=True))

--- Full Cross-Validation Results ---
   fold     auroc     auprc       acc
0     1  0.926316  0.936316  0.887417
1     2  0.913509  0.913556  0.834437
2     3  0.939123  0.935705  0.887417
3     4  0.958772  0.962869  0.894040
4     5  0.954035  0.955006  0.867550
5     6  0.948070  0.951950  0.894040
6     7  0.927895  0.916011  0.854305
7     8  0.947719  0.951629  0.880795
8     9  0.957368  0.950321  0.907285
9    10  0.923684  0.904588  0.867550
--------------------------------------------------
--- Overall Average Scores ---
fold     5.500000
auroc    0.939649
auprc    0.937795
acc      0.877483
dtype: float64


**📄 Code Explanation — Displaying Cross-Validation Results**

import pandas as pd
print(pd.read_csv("dreamwalk_cache/clean_cv_results.csv"))

This code is used to load and display the cross-validation results that were previously saved during the DREAMwalk evaluation stage.

 **1. Importing Pandas**

import pandas as pd
Imports the Pandas library, which is used for handling structured data in tabular form.
Pandas provides the DataFrame object, making it easy to read, organize, and analyze datasets such as CSV files.

**2. Reading the Results File**

pd.read_csv("dreamwalk_cache/clean_cv_results.csv")
Reads the file clean_cv_results.csv from the dreamwalk_cache directory.
This file contains the evaluation results generated during the cross-validation phase of the DREAMwalk pipeline.
pd.read_csv() loads the CSV file into a DataFrame, where the data is organized into rows and columns for easy inspection.

 **3. Printing the DataFrame**

print(...)
Displays the contents of the DataFrame directly in the console or notebook output.
This allows quick inspection of the performance metrics obtained from each cross-validation fold.

 **What the File Typically Contains**

The clean_cv_results.csv file usually stores one row per cross-validation fold, along with the corresponding evaluation metrics. Common columns include:

fold → the fold number in cross-validation
auroc → Area Under the ROC Curve
auprc → Area Under the Precision–Recall Curve
acc → classification accuracy

**Purpose of This Code**

This snippet is mainly used for reviewing model performance after training and evaluation. It provides a simple way to inspect how the DREAMwalk model performed across all folds and compare the consistency of its predictive results.